# Radar Imobiliário de Goiás — Projeto Final N2
Big Data e Data Science · FATESG 2026/02 · Hugo Vilhena

Fluxo: **R1** dados → **R2** dashboard → **R3** modelos. A base consolidada está em `dados/dados_municipios_goias.csv` (construída por `src/build_data.py` a partir das 7 tabelas IBGE da N1). Execute este notebook com o CSV na mesma pasta; a pasta `figuras` é criada se faltar.

In [1]:
import os; os.makedirs('figuras', exist_ok=True)
import pandas as pd
df = pd.read_csv('dados_municipios_goias.csv', index_col=0)
print(df.shape); df.head()

(246, 26)


,nome,regiao_imediata,regiao_intermediaria,pop_2024,pop_2025,pib_2021,pib_2022,pib_2023,va_agro,va_ind,...,share_ind,share_serv,share_adm,pib_pc_2023,pib_pc_2021,cresc_pop_24_25,cresc_pib_21_23,moradores_por_dom,hab_por_dom_2025,log_pop
cod,,,,,,,,,,,,,,,,,,,,,
5200050,Abadia de Goiás,Goiânia,Goiânia,21165,22052,353748,405353,477983,14325,79644,...,0.256647,0.501661,0.195530,22583.652256,16713.819986,0.041909,0.351196,3.004495,3.540215,9.960104
5200100,Abadiânia,Anápolis,Goiânia,17638,17738,445369,544312,633025,81208,56946,...,0.142435,0.420609,0.233837,35889.840118,25250.538610,0.005670,0.421349,2.746511,2.878611,9.777811
5200134,Acreúna,Palmeiras de Goiás,São Luís de Montes Belos - Iporá,21995,22079,1051903,1456803,1188179,480043,71132,...,0.072732,0.317681,0.118747,54020.413730,47824.641964,0.003819,0.129552,2.699164,2.795518,9.998570
5200159,Adelândia,Inhumas - Itaberaí - Anicuns,Goiânia,2297,2282,48810,53664,59486,11914,3546,...,0.076021,0.261743,0.406817,25897.257292,21249.455812,-0.006530,0.218726,2.492341,2.496718,7.739359
5200175,Água Fria de Goiás,Flores de Goiás,Luziânia - Águas Lindas de Goiás,4991,4979,533256,616437,476288,389836,20266,...,0.039419,0.137191,0.065133,95429.372871,106843.518333,-0.002404,-0.106830,2.732888,2.770729,8.515392


## R1 — Qualidade dos dados

In [2]:
print('nulos:', int(df.isna().sum().sum()), '| nomes duplicados:', int(df.nome.duplicated().sum()))
print('VA total / PIB 2021 (mediana):', round((df.va_total/df.pib_2021).median(),3))
df[df.va_ind<0][['nome','va_ind','pib_2021','pib_2023']]  # outlier excluído

nulos: 0 | nomes duplicados: 0
VA total / PIB 2021 (mediana): 0.933


,nome,va_ind,pib_2021,pib_2023
cod,,,,
5204250,Cachoeira Dourada,-477324,69505,890576


## R2 — Dashboard (4 gráficos + versão interativa dashboard.html)

In [3]:
"""Dashboard N2 - Radar Imobiliario de Goias (IBGE). Gera PNGs (README) e dashboard.html interativo (Plotly)."""
import numpy as np, pandas as pd, matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt, plotly.graph_objects as go
from plotly.subplots import make_subplots
df = pd.read_csv("dados_municipios_goias.csv", index_col=0); d = df[df.va_ind>=0].copy()
d["regiao"] = d.regiao_intermediaria.str.replace(" - .*","",regex=True).str.replace(" de Montes Belos","")
reg_cor = dict(zip(sorted(d.regiao.unique()), ["#1d5fd6","#0e7a8a","#7c3aed","#b3801e","#c0355a","#0e7a3f"]))
plt.rcParams.update({"axes.spines.top":False,"axes.spines.right":False,"font.size":10})
# 1) populacao x PIB per capita
fig,ax=plt.subplots(figsize=(8,5))
for r,g in d.groupby("regiao"): ax.scatter(g.pop_2024,g.pib_pc_2023,s=28,alpha=.75,color=reg_cor[r],label=r)
ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlabel("População 2024 (escala log)"); ax.set_ylabel("PIB per capita 2023, R$ (escala log)")
ax.set_title("Municípios maiores não são os mais ricos por habitante"); ax.legend(fontsize=8,frameon=False)
for n in ["Goiânia","Catalão","Anápolis"]:
    r=d[d.nome==n].iloc[0]; ax.annotate(n,(r.pop_2024,r.pib_pc_2023),fontsize=8,xytext=(5,5),textcoords="offset points")
plt.tight_layout(); plt.savefig("figuras/01_populacao_x_pib_per_capita.png",dpi=150); plt.close()
# 2) estrutura economica por regiao
s=d.groupby("regiao")[["va_agro","va_ind","va_serv","va_adm"]].sum(); s=s.div(s.sum(axis=1),axis=0)*100
s.columns=["Agropecuária","Indústria","Serviços","Adm. pública"]; s=s.loc[s["Adm. pública"].sort_values().index]
ax=s.plot.barh(stacked=True,figsize=(8,4.5),color=["#0e7a3f","#b3801e","#1d5fd6","#c0355a"],width=.7)
ax.set_xlabel("% do valor adicionado (2021)"); ax.set_ylabel(""); ax.set_title("Estrutura econômica por região intermediária")
ax.legend(ncol=4,fontsize=8,frameon=False,loc="upper center",bbox_to_anchor=(.5,-.15)); plt.tight_layout(); plt.savefig("figuras/02_estrutura_economica_regiao.png",dpi=150); plt.close()
# 3) top 15 PIB per capita
t=d.nlargest(15,"pib_pc_2023").sort_values("pib_pc_2023")
fig,ax=plt.subplots(figsize=(8,5)); ax.barh(t.nome,t.pib_pc_2023/1000,color="#1d5fd6"); ax.set_xlabel("PIB per capita 2023 (R$ mil)")
ax.set_title("15 municípios com maior PIB per capita"); plt.tight_layout(); plt.savefig("figuras/03_top15_pib_per_capita.png",dpi=150); plt.close()
# 4) correlacoes
cols=["log_pop","share_agro","share_ind","share_serv","share_adm","moradores_por_dom","cresc_pop_24_25"]
c=d.assign(ln_pib_pc=np.log(d.pib_pc_2023))[cols+["ln_pib_pc"]].corr()
c.index=c.columns=["ln pop","% agro","% indústria","% serviços","% adm. púb.","moradores/dom","cresc. pop","ln PIB pc"]
fig,ax=plt.subplots(figsize=(7,5.8)); im=ax.imshow(c,cmap="RdBu_r",vmin=-1,vmax=1)
ax.set_xticks(range(len(c))); ax.set_xticklabels(c.columns,rotation=40,ha="right"); ax.set_yticks(range(len(c))); ax.set_yticklabels(c.index)
for i in range(len(c)):
    for j in range(len(c)): ax.text(j,i,f"{c.iloc[i,j]:.2f}",ha="center",va="center",fontsize=8)
ax.set_title("Correlação entre variáveis"); plt.colorbar(im,fraction=.04); plt.tight_layout(); plt.savefig("figuras/04_correlacao.png",dpi=150); plt.close()
# dashboard interativo
f=make_subplots(rows=2,cols=2,subplot_titles=("População x PIB per capita (log)","Estrutura econômica por região (% VA)","Top 15 PIB per capita (R$ mil)","Correlação com ln(PIB per capita)"),vertical_spacing=.16)
for r,g in d.groupby("regiao"):
    f.add_trace(go.Scatter(x=g.pop_2024,y=g.pib_pc_2023,mode="markers",name=r,text=g.nome,marker=dict(color=reg_cor[r],size=7,opacity=.8),hovertemplate="%{text}<br>pop %{x:,}<br>R$ %{y:,.0f}"),1,1)
for col,cor in zip(s.columns,["#0e7a3f","#b3801e","#1d5fd6","#c0355a"]):
    f.add_trace(go.Bar(y=s.index,x=s[col],name=col,orientation="h",marker_color=cor),1,2)
f.add_trace(go.Bar(y=t.nome,x=t.pib_pc_2023/1000,orientation="h",marker_color="#1d5fd6",showlegend=False),2,1)
cc=c["ln PIB pc"].drop("ln PIB pc").sort_values()
f.add_trace(go.Bar(y=cc.index,x=cc.values,orientation="h",marker_color=["#c0355a" if v<0 else "#0e7a8a" for v in cc],showlegend=False),2,2)
f.update_xaxes(type="log",row=1,col=1); f.update_yaxes(type="log",row=1,col=1); f.update_layout(barmode="stack",height=900,title="Radar Imobiliário de Goiás - Dashboard N2 (IBGE)",template="simple_white",legend=dict(orientation="h",y=-.08))
f.update_layout(**{"barmode":"stack"}); f.write_html("dashboard.html",include_plotlyjs="cdn"); print("ok")


ok


## R3 — Modelagem: Ridge × Random Forest × Gradient Boosting

In [4]:
"""Projeto Final N2 - Big Data e Data Science (FATESG 2026/02) - Hugo Vilhena
Modelagem preditiva: PIB per capita 2023 dos municipios de Goias (regressao)."""
import numpy as np, pandas as pd, json, matplotlib
matplotlib.use("Agg"); import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.inspection import permutation_importance

df = pd.read_csv("dados_municipios_goias.csv", index_col=0)
# --- Tratamento: outlier estrutural (VA industrial negativo, usina hidreletrica) ---
outl = df.va_ind < 0
print("Removidos do treino:", df.loc[outl,"nome"].tolist())
d = df[~outl].copy()
d["share_total_check"] = d[["share_agro","share_ind","share_serv","share_adm"]].sum(axis=1)
assert np.allclose(d.share_total_check, 1)
d["y"] = np.log(d.pib_pc_2023)                      # alvo em log (distribuicao muito assimetrica)
d = d.join(pd.get_dummies(d.regiao_intermediaria, prefix="reg", dtype=float))
feats = ["log_pop","share_agro","share_ind","share_serv","share_adm","moradores_por_dom","cresc_pop_24_25"] + [c for c in d if c.startswith("reg_")]
X, y = d[feats], d.y
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)
models = {
 "Ridge (linear)": make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
 "Random Forest": RandomForestRegressor(n_estimators=400, min_samples_leaf=3, random_state=42),
 "Gradient Boosting": GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=3, subsample=0.8, random_state=42),
}
cv = KFold(5, shuffle=True, random_state=42)
rows, preds = [], {}
for n, m in models.items():
    s = cross_validate(m, Xtr, ytr, cv=cv, scoring=("r2","neg_mean_absolute_error"))
    m.fit(Xtr, ytr); p = m.predict(Xte); preds[n] = p
    rows.append(dict(modelo=n, r2_cv=s["test_r2"].mean(), r2_cv_std=s["test_r2"].std(),
        r2_teste=r2_score(yte,p), mae_log_teste=mean_absolute_error(yte,p),
        rmse_log_teste=mean_squared_error(yte,p)**.5,
        mae_reais_teste=mean_absolute_error(np.exp(yte),np.exp(p))))
res = pd.DataFrame(rows).round(3); print(res.to_string(index=False))
res.to_csv("resultados_modelos.csv", index=False, encoding="utf-8-sig")
# baseline ingenuo
base = np.full(len(yte), ytr.mean()); print("baseline R2:", round(r2_score(yte,base),3), "MAE R$:", round(mean_absolute_error(np.exp(yte),np.exp(base))))
best = res.sort_values("r2_teste").iloc[-1].modelo; print("melhor:", best)
# importancia por permutacao no melhor modelo
pi = permutation_importance(models[best], Xte, yte, n_repeats=30, random_state=42, scoring="r2")
imp = pd.Series(pi.importances_mean, index=feats).sort_values()
imp.to_csv("importancia_variaveis.csv", header=["importancia"]); print(imp.tail(6).round(3))
# --- figuras do modelo ---
fig, ax = plt.subplots(1,3, figsize=(15,4.4), sharex=True, sharey=True)
for a,(n,p) in zip(ax,preds.items()):
    a.scatter(yte,p,alpha=.7,color="#1d5fd6"); lim=[yte.min()-.1,yte.max()+.1]; a.plot(lim,lim,"k--",lw=1)
    a.set_title(f"{n}\nR² teste = {r2_score(yte,p):.2f}"); a.set_xlabel("ln(PIB per capita) real")
ax[0].set_ylabel("ln(PIB per capita) previsto"); plt.tight_layout(); plt.savefig("figuras/05_previsto_vs_real.png", dpi=150); plt.close()
fig, ax = plt.subplots(figsize=(7,4.2)); imp.tail(10).plot.barh(ax=ax,color="#0e7a8a")
ax.set_title(f"Importância das variáveis (permutação) - {best}"); ax.set_xlabel("queda no R² ao embaralhar a variável")
plt.tight_layout(); plt.savefig("figuras/06_importancia_variaveis.png", dpi=150); plt.close()
json.dump({"n_treino":len(Xtr),"n_teste":len(Xte),"melhor":best,"feats":feats}, open("meta.json","w"))


Removidos do treino: ['Cachoeira Dourada']


           modelo  r2_cv  r2_cv_std  r2_teste  mae_log_teste  rmse_log_teste  mae_reais_teste
   Ridge (linear)  0.764      0.067     0.812          0.174           0.241        10143.827
    Random Forest  0.783      0.033     0.875          0.150           0.197         8682.471
Gradient Boosting  0.807      0.048     0.893          0.133           0.182         8525.355
baseline R2: -0.036 MAE R$: 23548
melhor: Gradient Boosting


share_serv         0.004
log_pop            0.024
cresc_pop_24_25    0.028
share_agro         0.032
reg_Rio Verde      0.044
share_adm          1.689
dtype: float64


**Conclusão:** Gradient Boosting teve o melhor R² no teste; a participação da administração pública no valor adicionado é, de longe, a variável mais importante. Detalhes e limitações no README.